In [1]:
import os
import shutil

# 1. Define Paths
WHEEL_DIR = "/kaggle/working/wheels_fast"
# This is the path to your uploaded dataset wheel
SOURCE_WHEEL = "/kaggle/input/datasets/pjleek/onnx-runtime126-311/onnxruntime-1.26.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"

# 2. Reset and Create Directory
if os.path.exists(WHEEL_DIR):
    shutil.rmtree(WHEEL_DIR)
os.makedirs(WHEEL_DIR, exist_ok=True)

# 3. Move the "Tool" into the workspace
print(f"Moving ONNX Runtime 1.26.0 from Dataset to {WHEEL_DIR}...")

if os.path.exists(SOURCE_WHEEL):
    # Copy the file to your destination folder
    dest_path = os.path.join(WHEEL_DIR, os.path.basename(SOURCE_WHEEL))
    shutil.copy(SOURCE_WHEEL, dest_path)
else:
    print(f"❌ ERROR: Source wheel not found at {SOURCE_WHEEL}")
    print("Please check if the Dataset is correctly attached to the notebook.")

# 4. Final Audit
print("\n--- Final Directory Audit ---")
downloaded_files = os.listdir(WHEEL_DIR)
if any("onnxruntime" in f for f in downloaded_files):
    print(f"✅ onnxruntime 1.26.0 successfully staged: {downloaded_files[0]}")
else:
    print("❌ ERROR: onnxruntime is MISSING.")

Moving ONNX Runtime 1.26.0 from Dataset to /kaggle/working/wheels_fast...

--- Final Directory Audit ---
✅ onnxruntime 1.26.0 successfully staged: onnxruntime-1.26.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl


In [2]:
%%writefile main.py
import math
import numpy as np
import os
import sys
import zipfile
import glob
from collections import defaultdict, namedtuple

# ============================================================
# 1. DEPENDENCY & ONNX SESSION RUNTIME INJECTOR
# ============================================================
try:
    AGENT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    AGENT_DIR = "/kaggle_simulations/agent"
    if not os.path.exists(AGENT_DIR): AGENT_DIR = os.getcwd()

LIB_DIR = os.path.join(AGENT_DIR, "local_lib")
os.makedirs(LIB_DIR, exist_ok=True)
sys.path.insert(0, LIB_DIR)

ort = None
try:
    import onnxruntime as ort
except ImportError:
    print("Extracting ONNXRuntime...", file=sys.stderr)
    wheel_paths = [
        os.path.join(AGENT_DIR, "wheels_fast"),
        "/kaggle_simulations/agent/wheels_fast",
        "/kaggle/input/datasets/pjleek/euro-step-v4/wheels_fast"
    ]
    for wp in wheel_paths:
        wheels = glob.glob(os.path.join(wp, "onnxruntime*.whl"))
        if wheels:
            with zipfile.ZipFile(wheels[0], 'r') as z:
                z.extractall(LIB_DIR)
            try:
                import onnxruntime as ort
                print("SUCCESS: ONNX loaded.", file=sys.stderr)
                break
            except Exception:
                pass

onnx_sessions = {'2p': None, '4p': None}
_logged_status = False

def initialize_sessions(agent_dir):
    paths = {
        '2p': [
            os.path.join(agent_dir, "2p/orbit_model.onnx"),
            "/kaggle_simulations/agent/2p/orbit_model.onnx",
            "/kaggle/input/datasets/pjleek/euro-step-v4/2p/orbit_model.onnx",
            "/kaggle/working/2p/orbit_model.onnx"
        ],
        '4p': [
            os.path.join(agent_dir, "4p/orbit_model.onnx"),
            "/kaggle_simulations/agent/4p/orbit_model.onnx",
            "/kaggle/input/datasets/pjleek/euro-step-v4/4p/orbit_model.onnx",
            "/kaggle/working/4p/orbit_model.onnx"
        ]
    }
    for mode in ['2p', '4p']:
        for path in paths[mode]:
            if os.path.exists(path):
                try:
                    onnx_sessions[mode] = ort.InferenceSession(path)
                    print(f"ONNX {mode} session initialized from {path}", file=sys.stderr)
                    break
                except Exception as e:
                    print(f"Failed to load {path}: {e}", file=sys.stderr)

initialize_sessions(AGENT_DIR)

# ============================================================
# 2. CONSTANTS & NAMED TUPLES
# ============================================================
CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R       = 10.0
SUN_SAFETY  = 0.5
MAX_SPEED   = 6.0
TOTAL_STEPS = 500

TRAPS = [
    (15, 83), (5, 67), (9, 76), (75, 75), (89, 61),
    (6, 28), (90, 68), (95, 69), (56, 9), (36, 18)
]

MIN_NN_SCORE = 0.45

Planet = namedtuple("Planet", ["id", "owner", "x", "y", "radius", "ships", "production"])
Fleet = namedtuple("Fleet", ["id", "owner", "x", "y", "angle", "from_planet_id", "ships"])

# ============================================================
# 3. STATIC DIMENSION MATRIX CONFIGURATIONS
# ============================================================
CONFIGS = {
    '2p': {
        'MAX_PLANETS': 32,
        'MAX_FLEETS': 62,
        'PLANET_FEATURES': 7,
        'FLEET_FEATURES': 5,
        'TOTAL_SIZE': 542
    },
    '4p': {
        'MAX_PLANETS': 40,
        'MAX_FLEETS': 100,
        'PLANET_FEATURES': 7,
        'FLEET_FEATURES': 5,
        'TOTAL_SIZE': 782
    }
}

# ============================================================
# 4. HIGH-PERFORMANCE VECTORIZER PIPELINE
# ============================================================
def observation_to_binary_vector(obs, player_id, is_4p):
    mode = '4p' if is_4p else '2p'
    cfg = CONFIGS[mode]
    
    max_planets = cfg['MAX_PLANETS']
    max_fleets = cfg['MAX_FLEETS']
    planet_feats = cfg['PLANET_FEATURES']
    fleet_feats = cfg['FLEET_FEATURES']
    
    vector = np.zeros(cfg['TOTAL_SIZE'], dtype=np.float32)
    
    is_dict = isinstance(obs, dict)
    get     = lambda key, default: obs.get(key, default) if is_dict else getattr(obs, key, default)

    step = get('step', 0)
    omega = get('angular_velocity', 0.03)
    planets_data = get('planets', [])
    fleets_data = get('fleets', [])

    vector[0] = float(step) / 500.0
    vector[1] = float(omega)
    
    sorted_planets = sorted(planets_data, key=lambda x: x[0])
    
    # Vectorize Planets
    idx = 2
    for i in range(max_planets):
        if i < len(sorted_planets):
            pid, owner, x, y, radius, ships, prod = sorted_planets[i][:7]
            
            owner_encoded = 0.0
            if owner == player_id:
                owner_encoded = 1.0
            elif owner != -1:
                owner_encoded = -1.0
                
            dist_to_sun = np.hypot(x - 50.0, y - 50.0)
            is_orbiting = 1.0 if (dist_to_sun + radius) < 50.0 else 0.0
            
            vector[idx]   = owner_encoded
            vector[idx+1] = float(x)
            vector[idx+2] = float(y)
            vector[idx+3] = float(radius)
            vector[idx+4] = float(ships)
            vector[idx+5] = float(prod)
            vector[idx+6] = is_orbiting
        else:
            vector[idx : idx+planet_feats] = 0.0
            
        idx += planet_feats

    # Vectorize Active Fleets
    for j in range(max_fleets):
        if j < len(fleets_data):
            _, f_owner, fx, fy, _, _, f_ships = fleets_data[j][:7]
            f_owner_encoded = 1.0 if f_owner == player_id else -1.0
            
            vector[idx]   = f_owner_encoded
            vector[idx+1] = float(fx)
            vector[idx+2] = float(fy)
            vector[idx+3] = float(f_ships)
            vector[idx+4] = 1.0
        else:
            vector[idx : idx+fleet_feats] = 0.0
            
        idx += fleet_feats
        
    return vector

# ============================================================
# 5. VECTORIZED STATE TRANSITION SIMULATOR
# ============================================================
class VectorStateSimulator:
    def __init__(self, state_vector, is_4p=False):
        self.state = state_vector.copy()
        mode = '4p' if is_4p else '2p'
        self.cfg = CONFIGS[mode]
        self.max_planets = self.cfg['MAX_PLANETS']
        self.max_fleets = self.cfg['MAX_FLEETS']
        self.planet_feats = self.cfg['PLANET_FEATURES']
        self.fleet_feats = self.cfg['FLEET_FEATURES']
        self.fleet_start = 2 + (self.max_planets * self.planet_feats)

    def step(self):
        omega = self.state[1]
        self.state[0] += (1.0 / 500.0)
        
        # Planet updates
        for i in range(self.max_planets):
            p_idx = 2 + (i * self.planet_feats)
            owner = self.state[p_idx]
            is_active = self.state[p_idx + 1] != 0.0 or self.state[p_idx + 2] != 0.0
            
            if not is_active:
                continue
                
            if owner != 0.0:
                prod = self.state[p_idx + 5]
                self.state[p_idx + 4] += prod
                
            is_orbiting = self.state[p_idx + 6]
            if is_orbiting == 1.0:
                x = self.state[p_idx + 1] - 50.0
                y = self.state[p_idx + 2] - 50.0
                
                cos_o = np.cos(omega)
                sin_o = np.sin(omega)
                
                self.state[p_idx + 1] = (x * cos_o - y * sin_o) + 50.0
                self.state[p_idx + 2] = (x * sin_o + y * cos_o) + 50.0
                
        return self.state

# ============================================================
# 6. PHYSICS & SCORING HEURISTICS
# ============================================================
def get_danger_heat(tx: float, ty: float) -> float:
    if not TRAPS:
        return 0.0
    min_d = min(math.hypot(tx - t[0], ty - t[1]) for t in TRAPS)
    return max(0.0, min(1.0, 1.0 - (min_d / 15.0)))

def fleet_speed(ships: int) -> float:
    if ships <= 1:
        return 1.0
    return 1.0 + (5.0) * (min(1.0, math.log(ships) / math.log(1000.0)) ** 1.5)

def point_to_segment_dist(px, py, x1, y1, x2, y2) -> float:
    dx, dy = x2 - x1, y2 - y1
    l2 = dx*dx + dy*dy
    if l2 == 0:
        return math.hypot(px - x1, py - y1)
    t = max(0.0, min(1.0, ((px - x1)*dx + (py - y1)*dy) / l2))
    return math.hypot(px - (x1 + t*dx), py - (y1 + t*dy))

def get_target_pos(tgt, turns: int, ang_vel: float, initial_planets: dict, comets: list, comet_ids: set):
    if tgt.id in comet_ids:
        for c in comets:
            if tgt.id in c.get("planet_ids", []):
                idx = c["planet_ids"].index(tgt.id)
                f_idx = c.get("path_index", 0) + turns
                if idx < len(c["paths"]) and 0 <= f_idx < len(c["paths"][idx]):
                    return tuple(c["paths"][idx][f_idx])
        return (tgt.x, tgt.y)

    init = initial_planets.get(tgt.id)
    if not init:
        return (tgt.x, tgt.y)

    r = math.hypot(init.x - CENTER_X, init.y - CENTER_Y)
    if r + init.radius >= 50.0:
        return (tgt.x, tgt.y)

    ang = math.atan2(tgt.y - CENTER_Y, tgt.x - CENTER_X) + ang_vel * turns
    return (CENTER_X + r * math.cos(ang), CENTER_Y + r * math.sin(ang))

def plan_flight(src, tgt, ships: int, ang_vel: float, initial_planets: dict, comets: list, comet_ids: set):
    speed = fleet_speed(ships)
    tx, ty = tgt.x, tgt.y
    eta = 0.0

    for _ in range(6):
        dist = math.hypot(tx - src.x, ty - src.y)
        flight_dist = max(0.0, dist - src.radius - tgt.radius - 0.1)
        eta = flight_dist / speed
        pos = get_target_pos(tgt, int(math.ceil(eta)), ang_vel, initial_planets, comets, comet_ids)
        if not pos:
            return None, 999, tgt.x, tgt.y
        tx, ty = pos

    angle = math.atan2(ty - src.y, tx - src.x)

    sx = src.x + math.cos(angle) * (src.radius + 0.1)
    sy = src.y + math.sin(angle) * (src.radius + 0.1)
    ex = sx + math.cos(angle) * (eta * speed)
    ey = sy + math.sin(angle) * (eta * speed)

    if point_to_segment_dist(CENTER_X, CENTER_Y, sx, sy, ex, ey) <= SUN_R + SUN_SAFETY:
        return None, 999, tx, ty

    return angle, int(math.ceil(eta)), tx, ty

def get_config_val(config, key, default):
    if config is None:
        return default
    if isinstance(config, dict):
        return config.get(key, default)
    return getattr(config, key, default)

def classify_live_bins(planets, initial_planets):
    total_planets = len(planets)
    if total_planets == 0:
        return 1, 1, 0

    avg_prod = sum(p.production for p in planets) / total_planets
    if avg_prod < 2.0: prod_bin = 0
    elif avg_prod < 3.0: prod_bin = 1
    elif avg_prod < 4.0: prod_bin = 2
    else: prod_bin = 3
    
    orbiting_count = 0
    for p in initial_planets.values():
        r = math.hypot(p.x - CENTER_X, p.y - CENTER_Y)
        if r + p.radius < 50.0:
            orbiting_count += 1
            
    orbiting_ratio = orbiting_count / total_planets
    if orbiting_ratio < 0.1: rot_bin = 0
    elif orbiting_ratio < 0.3: rot_bin = 1
    elif orbiting_ratio < 0.6: rot_bin = 2
    else: rot_bin = 3
    
    max_prod = max(p.production for p in planets) if planets else 5
    large_planets = [p for p in planets if p.production == max_prod]
    large_orbiting = 0
    for p in large_planets:
        r = math.hypot(p.x - CENTER_X, p.y - CENTER_Y)
        if r + p.radius < 50.0:
            large_orbiting += 1
            
    size_bin = 1 if (len(large_planets) > 0 and (large_orbiting / len(large_planets)) >= 0.5) else 0
    
    return prod_bin, rot_bin, size_bin

# ============================================================
# NEW: SYNCHRONIZED PREDICTIVE DISTANCE SOLVER
# ============================================================
def get_predicted_distance(src, tgt, ang_vel, initial_planets, comets, comet_ids, avail_ships):
    """
    Runs an iterative convergence loop to calculate exactly where the target 
    will be upon intercept, avoiding flat coordinate discrepancies [2].
    """
    speed = fleet_speed(avail_ships)
    tx, ty = tgt.x, tgt.y
    eta = 0.0
    
    for _ in range(6):
        dist = math.hypot(tx - src.x, ty - src.y)
        flight_dist = max(0.0, dist - src.radius - tgt.radius - 0.1)
        eta = flight_dist / speed
        
        pos = get_target_pos(tgt, int(math.ceil(eta)), ang_vel, initial_planets, comets, comet_ids)
        if not pos: 
            break
        tx, ty = pos
        
    return math.hypot(tx - src.x, ty - src.y), int(math.ceil(eta))

# ============================================================
# DYNAMIC EARLY AGGRESSION ATTACK CUSHION
# ============================================================
def ships_needed_for_takeover(tgt_ships, tgt_prod, tt, owner, step, target_prod, margin=1.05):
    if step <= 60 and target_prod >= 4:
        margin = 1.02
        
    if owner == -1:
        return int(tgt_ships * margin) + 1
    growth = tgt_prod * tt
    return int((tgt_ships + growth) * margin) + 1

# ============================================================
# 7. UNIFIED ENDER-LOGISTICS PRIORITY SCORER
# ============================================================
def compute_priority_score(src, tgt, tt, phase, player, active_enemy_planets, prod_bin, rot_bin, size_bin, vx, vy, t100_normalized):
    garrison = tgt.ships
    production = tgt.production
    
    roi = (production * 20.0) / (garrison + 3.0)
    time_discount = 1.0 / (1.0 + 0.005 * (tt ** 2))
    score = roi * time_discount * 15.0
    
    if tgt.owner == -1:
        score += 15.0
    else:
        if tgt.owner != player:
            if src.ships > garrison + 2:
                score += (production * 35.0) / (garrison + 1.0)
            else:
                score += 5.0

    tgt_is_static = (math.hypot(tgt.x - CENTER_X, tgt.y - CENTER_Y) + tgt.radius >= 50.0)
    if tgt_is_static:
        score += max(0.0, 15.0 * (1.0 - (garrison / 45.0)))
    else:
        score -= 4.0

    if phase == 'expand' and tgt.owner == -1:
        score += 15.0

    return score

# ============================================================
# 8. MAIN AGENT LOOP
# ============================================================
def agent(obs, config=None):
    global _logged_status
    
    is_dict = isinstance(obs, dict)
    get     = lambda key, default: obs.get(key, default) if is_dict else getattr(obs, key, default)

    player          = get("player", 0)
    step            = get("step", 0)
    planets         = [Planet(*p) for p in get("planets", [])]
    ang_vel         = get("angular_velocity", 0.0)
    initial_planets = {Planet(*p).id: Planet(*p) for p in get("initial_planets", [])}
    comets          = get("comets", [])
    comet_ids       = set(get("comet_planet_ids", []))
    raw_fleets      = get("fleets", [])
    fleets          = [Fleet(*f) for f in raw_fleets]

    my_planets      = [p for p in planets if p.owner == player]
    enemy_planets   = [p for p in planets if p.owner not in (-1, player)]
    neutral_planets = [p for p in planets if p.owner == -1]

    if not my_planets:
        return []

    # Initialize persistent state registers
    if not hasattr(agent, "mission_registry"):
        agent.mission_registry = {}
        agent.past_action_count = 0

    # Prune stale missions
    current_missions = {}
    for src_id, mission in agent.mission_registry.items():
        if step < mission["expected_arrival"]:
            current_missions[src_id] = mission
    agent.mission_registry = current_missions

    my_ships       = sum(p.ships for p in my_planets)
    my_prod        = sum(p.production for p in my_planets)
    enemy_prod     = sum(p.production for p in enemy_planets)
    prod_ratio     = my_prod / max(1, enemy_prod)

    prod_bin, rot_bin, size_bin = classify_live_bins(planets, initial_planets)

    active_enemy_planets = len(enemy_planets)
    my_planet_count = len(my_planets)

    # Detect Mode Strategy Gating
    n_players = float(get_config_val(config, "num_players", 2))
    is_4p = n_players > 2
    safety_buffer = 8 if is_4p else 1

    # Active Flight Tracking
    friendly_en_route = defaultdict(int)
    for f in fleets:
        if f.owner == player:
            best_dest = None
            closest_alignment = 1e9
            for p in planets:
                if p.id == f.from_planet_id:
                    continue
                expected_angle = math.atan2(p.y - f.y, p.x - f.x)
                angle_diff = abs((f.angle - expected_angle + math.pi) % (2 * math.pi) - math.pi)
                if angle_diff < 0.15:
                    d = math.hypot(p.x - f.x, p.y - f.y)
                    if d < closest_alignment:
                        closest_alignment = d
                        best_dest = p
            if best_dest:
                friendly_en_route[best_dest.id] += f.ships

    # Enforce global pacing limits
    if step < 15 and agent.past_action_count >= (4 if is_4p else 3):
        return []

    # ONNX GATED POLICY SELECTOR
    mode_str = '4p' if is_4p else '2p'
    onnx_session = onnx_sessions[mode_str]
    
    if onnx_session is not None:
        try:
            state_vector = observation_to_binary_vector(obs, player, is_4p)
            model_input = state_vector.reshape(1, -1)
            predictions = onnx_session.run(None, {'input': model_input})[0][0]
        except Exception:
            pass

    if step < 50 and len(my_planets) < 3:
        phase = 'expand'
    elif my_ships > 120 and len(my_planets) < 4 and enemy_planets:
        phase = 'rush'
    elif prod_ratio > 3.0 and my_ships > 80:
        phase = 'crush'
    else:
        phase = 'balanced'

    # ============================================================
    # RE-ARCHITECTED GLOBAL MANIFEST BOOKING POOL (Pass 1)
    # Scores ALL targets before applying budget/flight filters,
    # preserving true global ranking so Pass 2 never skips past
    # a premium secondary target due to a Ghost Rank deletion.
    # ============================================================
    all_potential_plays = []
    
    for src in my_planets:
        if src.id in agent.mission_registry:
            continue
            
        avail_ships = src.ships - (safety_buffer + 5)
        if avail_ships < 5:
            continue

        neutral_dists = [math.hypot(p.x - src.x, p.y - src.y) for p in planets if p.owner == -1]
        min_neutral_dist = min(neutral_dists) if neutral_dists else 100.0

        for tgt in planets:
            if tgt.id == src.id or tgt.owner == player:
                continue

            # 1. IMMEDIATE SPATIAL VECTOR EVALUATION
            pred_dist, eta = get_predicted_distance(src, tgt, ang_vel, initial_planets, comets, comet_ids, avail_ships)

            pos_t1 = get_target_pos(tgt, 1, ang_vel, initial_planets, comets, comet_ids)
            vx = (pos_t1[0] - tgt.x) if pos_t1 else 0.0
            vy = (pos_t1[1] - tgt.y) if pos_t1 else 0.0

            S_owned = sum(p.ships for p in my_planets)
            P_owned = sum(p.production for p in my_planets)
            t100 = (100.0 - S_owned + tgt.ships + tgt.production * eta) / max(1.0, float(P_owned + tgt.production))

            tgt_is_static = (math.hypot(tgt.x - CENTER_X, tgt.y - CENTER_Y) + tgt.radius >= 50.0)
            if tgt_is_static:
                t100 -= max(0.0, 8.0 * (1.0 - (tgt.ships / 40.0)))
            t100_normalized = min(1.0, max(0.0, t100 / 150.0))

            # Calculate raw structural value BEFORE pocket/budget constraints wipe it out
            raw_priority_score = compute_priority_score(
                src, tgt, eta, phase, player, active_enemy_planets,
                prod_bin, rot_bin, size_bin, vx, vy, t100_normalized
            )

            # 2. EVALUATE BUDGET & FLIGHT CLEARANCE
            already_sent = friendly_en_route[tgt.id]
            estimated_garrison = tgt.ships + (tgt.production * eta if tgt.owner != -1 else 0)

            if already_sent >= estimated_garrison:
                continue

            cost = ships_needed_for_takeover(estimated_garrison, tgt.production, eta, tgt.owner, step, tgt.production)

            is_double_reverse = False
            best_sibling = None
            sibling_cost = 0
            is_executable = True

            # Affordability Verification Gate
            if cost > avail_ships:
                for sibling in my_planets:
                    if sibling.id == src.id or sibling.id in agent.mission_registry:
                        continue
                    sib_avail = sibling.ships - (safety_buffer + 5)
                    if sib_avail < 5:
                        continue
                    _, sib_eta = get_predicted_distance(sibling, tgt, ang_vel, initial_planets, comets, comet_ids, sib_avail)
                    if abs(eta - sib_eta) <= 1.5 and (avail_ships + sib_avail) >= cost:
                        is_double_reverse = True
                        best_sibling = sibling
                        sibling_cost = max(5, cost - avail_ships + 1)
                        break
                if not is_double_reverse:
                    is_executable = False

            # Environmental Flight Clearance
            angle, _, _, _ = plan_flight(src, tgt, cost if not is_double_reverse else (cost - sibling_cost), ang_vel, initial_planets, comets, comet_ids)
            if angle is None:
                is_executable = False
            else:
                ex = src.x + math.cos(angle) * pred_dist
                ey = src.y + math.sin(angle) * pred_dist
                if point_to_segment_dist(CENTER_X, CENTER_Y, src.x, src.y, ex, ey) <= SUN_R + SUN_SAFETY:
                    is_executable = False

            sib_angle = None
            if is_double_reverse and is_executable:
                sib_angle, _, _, _ = plan_flight(best_sibling, tgt, sibling_cost, ang_vel, initial_planets, comets, comet_ids)
                if sib_angle is None:
                    is_executable = False

            max_eta = 80 if phase in ('crush', 'aggressive') else 55
            if eta > max_eta:
                is_executable = False

            if step <= 30 and pred_dist > min_neutral_dist + 12.0:
                is_executable = False

            # Append EVERYTHING to preserve global ranking integrity
            all_potential_plays.append({
                'src_id': src.id,
                'tgt_id': tgt.id,
                'cost': cost,
                'angle': angle,
                'eta': eta,
                'score': raw_priority_score,
                'executable': is_executable,
                'double_reverse': is_double_reverse,
                'sib_id': best_sibling.id if is_double_reverse else None,
                'sib_angle': sib_angle,
                'sib_cost': sibling_cost
            })

    # Sort the global manifest by true strategic reward
    all_potential_plays.sort(key=lambda x: x['score'], reverse=True)

    # --- PASS 2: THE STRICT EXECUTION GATE ---
    # Non-executable top plays consume a slot so the engine transitions
    # to Rank 2/3 instead of dropping off a cliff to Rank 6+.
    moves = []
    targeted = set()
    booked_sources = defaultdict(int)

    executable_count = 0
    for play in all_potential_plays:
        if executable_count >= 3:
            break  # Enforce standard Top 3 target window

        src_id = play['src_id']
        tgt_id = play['tgt_id']
        cost = play['cost']

        if tgt_id in targeted:
            continue

        # Premium target blocked or unaffordable: count the slot, skip execution
        if not play['executable']:
            executable_count += 1
            continue

        src_planet = next((p for p in my_planets if p.id == src_id), None)
        if not src_planet:
            continue

        avail_ships = src_planet.ships - (safety_buffer + 5) - booked_sources[src_id]

        if play['double_reverse']:
            sib_id = play['sib_id']
            sib_planet = next((p for p in my_planets if p.id == sib_id), None)
            sib_avail = sib_planet.ships - (safety_buffer + 5) - booked_sources[sib_id] if sib_planet else 0

            if avail_ships >= (cost - play['sib_cost']) and sib_avail >= play['sib_cost']:
                moves.append([src_id, play['angle'], cost - play['sib_cost']])
                moves.append([sib_id, play['sib_angle'], play['sib_cost']])

                booked_sources[src_id] += (cost - play['sib_cost'])
                booked_sources[sib_id] += play['sib_cost']
                targeted.add(tgt_id)

                agent.mission_registry[src_id] = {"target_id": tgt_id, "expected_arrival": step + play['eta']}
                agent.mission_registry[sib_id] = {"target_id": tgt_id, "expected_arrival": step + play['eta']}
                agent.past_action_count += 2
                return moves
        else:
            if avail_ships >= cost:
                moves.append([src_id, play['angle'], cost])
                booked_sources[src_id] += cost
                targeted.add(tgt_id)

                agent.mission_registry[src_id] = {"target_id": tgt_id, "expected_arrival": step + play['eta']}
                agent.past_action_count += 1
                return moves

    return []  # Hold the ball, collect interest, and build mass

Writing main.py


In [3]:
import tarfile
import shutil
import os

working_dir = "/kaggle/working"
submission_path = os.path.join(working_dir, "submission.tar.gz")

# Unpredictable mount variations across Kaggle notebook updates
dataset_search_paths = [
    "/kaggle/input/datasets/pjleek/euro-step-v4",
    "/kaggle/input/euro-step-v4",
    "/kaggle/input/pjleek/euro-step-v4",
    "/kaggle/working" 
]

def find_scoped_file(mode, filename):
    """
    Looks specifically inside mode subdirectories (e.g., '2p' or '4p') 
    across all potential dataset mount coordinates.
    """
    for base_path in dataset_search_paths:
        # Check both modern structured mounts (base/2p/file) and legacy fallbacks
        full_path = os.path.join(base_path, mode, filename)
        if os.path.exists(full_path):
            return full_path
    return None

print("🔍 Beginning Dual-Engine ONNX Staging...")

# Staging locations inside our temporary build directory
modes = ['2p', '4p']
staged_files = {m: {'onnx': None, 'data': None} for m in modes}

for mode in modes:
    print(f"\n[{mode.upper()} Pipeline]")
    
    # Locate files natively scoped within their respective game-type namespaces
    onnx_src = find_scoped_file(mode, "orbit_model.onnx")
    onnx_data_src = find_scoped_file(mode, "orbit_model.onnx.data")
    
    # Establish staging directory mirror path in local workspace
    mode_stage_dir = os.path.join(working_dir, mode)
    os.makedirs(mode_stage_dir, exist_ok=True)
    
    if not onnx_src:
        print(f"   ❌ CRITICAL ERROR: Could not find {mode}/orbit_model.onnx!")
    else:
        dest_path = os.path.join(mode_stage_dir, "orbit_model.onnx")
        if onnx_src != dest_path:
            shutil.copy(onnx_src, dest_path)
        staged_files[mode]['onnx'] = dest_path
        print(f"   ✅ Staged Model: {onnx_src} -> {mode}/")
        
    if onnx_data_src:
        dest_data_path = os.path.join(mode_stage_dir, "orbit_model.onnx.data")
        if onnx_data_src != dest_data_path:
            shutil.copy(onnx_data_src, dest_data_path)
        staged_files[mode]['data'] = dest_data_path
        print(f"   ✅ Staged Weights Tensor: {onnx_data_src} -> {mode}/")
    else:
        print(f"   ℹ️ Note: No separate external data weights found for {mode} (Model fits directly in graph proto).")

print("\n📦 Compiling Structural submission.tar.gz Archive...")
try:
    with tarfile.open(submission_path, "w:gz") as tar:
        # 1. Add main.py agent core to the root of the file bundle
        main_py_path = os.path.join(working_dir, "main.py")
        if os.path.exists(main_py_path):
            tar.add(main_py_path, arcname="main.py")
            print("   ➕ Added: main.py")
        else:
            print("   ❌ CRITICAL ERROR: main.py is missing from your working workspace root!")
            
        # 2. Add any supporting library scripts if present (e.g., simulator.py, vectorizer.py)
        for utility_script in ["simulator.py", "vectorizer.py"]:
            util_path = os.path.join(working_dir, utility_script)
            if os.path.exists(util_path):
                tar.add(util_path, arcname=utility_script)
                print(f"   ➕ Added: {utility_script}")

        # 3. Add Hierarchical Mode Directories preserving the 2p/ and 4p/ layout
        for mode in modes:
            mode_stage_dir = os.path.join(working_dir, mode)
            
            # Pack orbit_model.onnx inside its specific mode folder structure
            if staged_files[mode]['onnx'] and os.path.exists(staged_files[mode]['onnx']):
                tar.add(staged_files[mode]['onnx'], arcname=f"{mode}/orbit_model.onnx")
                print(f"   ➕ Added: {mode}/orbit_model.onnx")
                
            # Pack orbit_model.onnx.data alongside the base architecture binary if found
            if staged_files[mode]['data'] and os.path.exists(staged_files[mode]['data']):
                tar.add(staged_files[mode]['data'], arcname=f"{mode}/orbit_model.onnx.data")
                print(f"   ➕ Added: {mode}/orbit_model.onnx.data")

        # 4. Include offline container dependencies wheel folder
        wheels_dest = os.path.join(working_dir, "wheels_fast")
        if os.path.exists(wheels_dest):
            tar.add(wheels_dest, arcname="wheels_fast")
            print("   ➕ Added: wheels_fast dependency wheelhouse")
        else:
            print("   ⚠️ WARNING: wheels_fast directory missing. Ensure your agent initializes runtime components successfully on the sandbox container.")
            
    print(f"\n🎉 SUCCESS! Hierarchical agent container ready for leaderboard upload: {submission_path}")
        
except Exception as e:
    print(f"\n❌ FAILED to generate valid submission file structure: {e}")

🔍 Beginning Dual-Engine ONNX Staging...

[2P Pipeline]
   ❌ CRITICAL ERROR: Could not find 2p/orbit_model.onnx!
   ℹ️ Note: No separate external data weights found for 2p (Model fits directly in graph proto).

[4P Pipeline]
   ❌ CRITICAL ERROR: Could not find 4p/orbit_model.onnx!
   ℹ️ Note: No separate external data weights found for 4p (Model fits directly in graph proto).

📦 Compiling Structural submission.tar.gz Archive...
   ➕ Added: main.py
   ➕ Added: wheels_fast dependency wheelhouse

🎉 SUCCESS! Hierarchical agent container ready for leaderboard upload: /kaggle/working/submission.tar.gz


In [4]:
import tarfile

# Define the path to your submission archive
submission_path = "/kaggle/working/submission.tar.gz"

try:
    with tarfile.open(submission_path, "r:gz") as tar:
        print(f"📦 Contents of {submission_path}:")
        for name in tar.getnames():
            print(f"  - {name}")
except Exception as e:
    print(f"❌ Error opening archive: {e}")

📦 Contents of /kaggle/working/submission.tar.gz:
  - main.py
  - wheels_fast
  - wheels_fast/onnxruntime-1.26.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
